In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Real topics from the Steve Jobs speech
topics_text = [
    "I'm honored to be with you today for your commencement from one of the finest universities in the world. And uh, this is the closest I've ever gotten to a college graduation. Truth be told, I never graduated from college.",
    "That's it. No big deal. Just three stories. The first story is about connecting the dots. I dropped out of Reed College after the first six months, but then stayed around as a drop-in for another eighteen months or so before I really quit. So why'd I drop out?",
    "So my parents, who were on a waiting list, got a call in the middle of the night asking, unexpected question, we've got an unplanned baby boy. Do you want him? They said, of course.",
    "My biological mother found out later that my mother had never graduated from college and that my father had never graduated from high school. She refused to sign the final adoption papers.",
    "Your work is going to fill a large part of your life. And the only way to be truly satisfied is to do what you believe is great work. And the only way to do great work is to love what you do.",
    "Stay hungry. Stay foolish. And I have always wished that for myself. And now, as you graduate to begin anew, I wish that for you. Stay hungry, stay foolish. Thank you all very much.",
]

def generate_title_v2(sentences_text: str, all_topics: list[str], max_words: int = 5) -> str:
    """Generate a short descriptive title from keywords."""
    vectorizer = TfidfVectorizer(stop_words="english", max_features=200)
    tfidf_matrix = vectorizer.fit_transform(all_topics)
    feature_names = vectorizer.get_feature_names_out()
    
    topic_idx = all_topics.index(sentences_text)
    scores = tfidf_matrix[topic_idx].toarray()[0]
    
    # Get top 3 keywords sorted by score
    top_indices = scores.argsort()[-3:][::-1]
    keywords = [feature_names[idx] for idx in top_indices if scores[idx] > 0]
    
    if not keywords:
        # Fallback: first few words of the text
        words = sentences_text.split()[:max_words]
        return " ".join(words) + "..."
    
    # Capitalize first keyword, join with " & " or ", "
    title = ", ".join(w.capitalize() for w in keywords[:3])
    return title

for i, topic in enumerate(topics_text):
    title = generate_title_v2(topic, topics_text)
    print(f"Topic {i+1}: {title}")

Topic 1: College, World, Gotten
Topic 2: Drop, Months, Reed
Topic 3: Got, List, Night
Topic 4: Mother, Graduated, Adoption
Topic 5: Work, Way, Great
Topic 6: Stay, Hungry, Foolish


In [ ]:
def generate_title_v4(sentences_text: str, all_topics: list[str], max_words: int = 6) -> str:
    """Best of both: keyword title if clean, sentence fragment if not."""
    vectorizer = TfidfVectorizer(stop_words="english", max_features=200)
    tfidf_matrix = vectorizer.fit_transform(all_topics)
    feature_names = vectorizer.get_feature_names_out()
    
    topic_idx = all_topics.index(sentences_text)
    scores = tfidf_matrix[topic_idx].toarray()[0]
    
    top_indices = scores.argsort()[-3:][::-1]
    keywords = [feature_names[idx] for idx in top_indices if scores[idx] > 0]
    
    if not keywords:
        words = sentences_text.split()[:max_words]
        return " ".join(words) + "..."
    
    # Find the sentence containing the top keyword
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(sentences_text)
    
    best_sentence = sentences[0]
    best_count = 0
    for sent in sentences:
        count = sum(1 for kw in keywords if kw in sent.lower())
        if count > best_count:
            best_count = count
            best_sentence = sent
    
    # If the sentence is short enough, use it as-is
    words = best_sentence.split()
    if len(words) <= max_words:
        return best_sentence
    
    # Try to find a shorter clause containing the keyword
    # Split on commas and periods to find a short fragment
    fragments = best_sentence.replace(". ", ", ").split(", ")
    for frag in fragments:
        frag = frag.strip()
        if any(kw in frag.lower() for kw in keywords) and len(frag.split()) <= max_words:
            return frag.rstrip(".")
    
    # Fallback: truncate the best sentence
    return " ".join(words[:max_words]) + "..."

for i, topic in enumerate(topics_text):
    title = generate_title_v4(topic, topics_text)
    print(f"Topic {i+1}: {title}")

Topic 1: And uh, this is the closest...
Topic 2: I dropped out of Reed College...
Topic 3: So my parents, who were on...
Topic 4: My biological mother found out later...
Topic 5: Your work is going to fill...
Topic 6: Stay hungry.
